# Novelty Search (Skeleton)
For use in the experiments of the Amorphous Fortress Narrative generation

### General Pseudocode
1. Create initial population of genomes
2. Evaluate each for fitness
3. Evaluate each for novelty against archive genomes
4. Add fit and novel genomes in archive
5. Select new parents of population from novelty archive
6. Mutate children and create new population
7. (Add random back in)
8. Repeat 2-7 for n generations

(Reference: [SimSim](https://github.com/lsoros/simsim/blob/master/simsim.cpp) and [Algorithm Definition](https://algorithmafternoon.com/novelty/novelty_search_algorithm/))

### Setup

In [1]:
# imports
import random
import numpy as np
import json
import spacy
import re
from datetime import datetime
import pyinflect
from sentence_transformers import SentenceTransformer
import torch

/Users/mcharit2/Desktop/Research/AF-Narrate/amorphous-fortress-narratives/af-narrate/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# set models for NLP tasks
nlp = spacy.load("en_core_web_sm")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

In [3]:
# import data
ALL_SUBJS = np.load('../bank_files/SUBJECTS_clean.npy', allow_pickle=True)
ALL_OBJS = np.load('../bank_files/OBJECTS_clean.npy', allow_pickle=True)
ALL_VERBS = np.load('../bank_files/VERBS_clean.npy', allow_pickle=True)
CN_GRAPH = json.load(open('../bank_files/full_word_graph_noweight.json'))

# convert to normal lists
ALL_SUBJS = [str(s) for s in ALL_SUBJS]
ALL_OBJS = [str(o) for o in ALL_OBJS]
ALL_VERBS = [str(v) for v in ALL_VERBS]

print(len(ALL_SUBJS), len(ALL_OBJS), len(ALL_VERBS))

print(random.choices(ALL_SUBJS, k=5), random.choices(ALL_OBJS, k=5), random.choices(ALL_VERBS, k=5))

# TODO: encode all subjects, objects, verbs using sentence transformer model for faster eval and lookup

5344 3191 1378
['audiophile', 'uncleanliness', 'horses', 'officer', 'illnesses'] ['resource', 'commitment', 'roulette', 'skin', 'restroom'] ['floored', 'augmented', 'consumed', 'rang', 'compared']


In [4]:
# convert all verbs to past tense
# def to_past_tense(verbs):
#     return [verb._.inflect("VBD") if verb._.inflect("VBD") is not None else verb.text for verb in nlp(' '.join(verbs))]

# ALL_VERBS = to_past_tense(ALL_VERBS)

In [5]:
# create verbs and verb encodings
AF_VERBS = ["moved", "died", "cloned", "took", "pushed", "added", "transformed", "blocked", "chased"]
all_af_verb_encs = st_model.encode(AF_VERBS)
af_verb_enc_dict = {AF_VERBS[i]: all_af_verb_encs[i] for i in range(len(AF_VERBS))}

In [6]:
# constants
MC_MUTATE_PERC = 0.25
ENT_MUTATE_PERC = 0.1
VERB_MUTATE_PERC = 0.25
POP_SIZE = 10
NUM_GENERATIONS = 20

In [7]:
# maps a log for usage in the novelty search
class AF_Story:
    def __init__(self, log_file):
        self.log_file = log_file
        with open(log_file, 'r') as f:
            self.og_text = [line.strip() for line in f.readlines()]
        self.ent_ids = self.find_spec_ents()        # dict of entities with subject/object designation
        self.ent_reps = self.get_ent_reps()

        self.ent_order, self.mc_ent = self.find_ents()      # list of all entities in the original text
        self.verb_set = self.find_verbs()          # list of tuples of (subject entity, verb)
        self.verb_order = [v[1] for v in self.verb_set.values()]    # list of all verbs in the original text


    def find_ents(self):
        ''' Find all AF entities in the original text. '''
        af_ents = []
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                af_ents.extend(match)

        # get highest occuring entity as main character
        random.shuffle(af_ents) # shuffle to avoid biasing first entity as MC
        mc_ent = max(set(af_ents), key = af_ents.count)
        return list(set(af_ents)), mc_ent

    def find_spec_ents(self):
        ''' Identifies entities and whether they are the subject or object in the sentence. '''
        af_ents = {}
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                for i in range(len(match)):
                    if i == 0:
                        af_ents[match[i]] = 'subject'
                    elif match[i] not in af_ents:
                        af_ents[match[i]] = 'object'
                    
        return af_ents
    
    def get_ent_reps(self):
        ''' Get the symbol representations of the entities in the story '''
        return list(set([e[1] for e in self.ent_ids.keys()]))
    
    def find_verbs(self):
        ''' Find all verbs in the original text '''
        af_verbs = {}
        for i, line in enumerate(self.og_text):
            subj_ent = re.findall(r'(\[.\..{4}\])', line)
            for verb in AF_VERBS:
                if verb in line:
                    af_verbs[i] = (subj_ent[0], verb)
                    break # only take the first verb found
        return af_verbs
    


In [8]:
# import stories
stupid_story = AF_Story('../logs/stupid_log.txt')
castle_story = AF_Story('../sifted_logs/sifted_castle2.txt')
zelda_story = AF_Story('../sifted_logs/sifted_zelda.txt')

In [9]:
# test
print("Ent IDs:\t" + str(castle_story.ent_ids))
print("Ent Reps:\t" + str(castle_story.ent_reps))
print("MC Ent:\t" + str(castle_story.mc_ent))
print("Ent Order:\t" + str(castle_story.ent_order))
print("Verb Set:\t" + str(castle_story.verb_set))
print("Verb Order:\t" + str(castle_story.verb_order))

Ent IDs:	{'[K.fc7c]': 'subject', '[T.d89f]': 'subject', '[T.73ab]': 'subject', '[=.23f6]': 'object', '[G.3305]': 'subject', '[$.1a6e]': 'object', '[T.76a3]': 'subject', '[$.6d7b]': 'object', '[$.9fe4]': 'object', '[=.2486]': 'object', '[$.1b0e]': 'object', '[=.0cc4]': 'object', '[$.0bdf]': 'object', '[K.7479]': 'subject', '[$.6666]': 'object', '[$.6924]': 'object', '[$.db0a]': 'object', '[$.4709]': 'object'}
Ent Reps:	['G', '$', 'K', 'T', '=']
MC Ent:	[T.73ab]
Ent Order:	['[=.2486]', '[$.6d7b]', '[$.4709]', '[$.1b0e]', '[T.76a3]', '[K.7479]', '[T.73ab]', '[$.6924]', '[$.db0a]', '[T.d89f]', '[G.3305]', '[K.fc7c]', '[$.9fe4]', '[$.6666]', '[$.0bdf]', '[$.1a6e]', '[=.23f6]', '[=.0cc4]']
Verb Set:	{0: ('[K.fc7c]', 'moved'), 1: ('[T.73ab]', 'moved'), 2: ('[K.fc7c]', 'moved'), 3: ('[T.d89f]', 'blocked'), 4: ('[T.73ab]', 'moved'), 5: ('[G.3305]', 'died'), 6: ('[T.73ab]', 'took'), 7: ('[T.76a3]', 'moved'), 8: ('[T.d89f]', 'moved'), 9: ('[T.73ab]', 'took'), 10: ('[K.fc7c]', 'moved'), 11: ('[K.f

In [10]:
# FOR USE WITH THE MUTATION / NOVELTY SEARCH
def get_assoc_dat(mc_ent):
    ''' Gets the associative data (related subjects, objects, and verbs) for a given entity representation '''
    if not mc_ent or mc_ent not in CN_GRAPH:
        return {}
    
    dat = CN_GRAPH[mc_ent]
    verbs = dat.keys()
    assoc_ents = []
    for v in verbs:
        assoc_ents.extend(dat[v])
    subj_ents = [e for e in assoc_ents if e in ALL_SUBJS]
    obj_ents = [e for e in assoc_ents if e in ALL_OBJS]
    subj_ents = list(set(subj_ents))  # remove duplicates
    obj_ents = list(set(obj_ents))    # remove duplicates
    return {'subj': subj_ents, 'obj': obj_ents, 'verbs': list(verbs)}


def get_assoc_verbs(ent):
    ''' Gets the associative verbs for a given entity representation '''
    if not ent or ent not in CN_GRAPH:
        return []
    dat = CN_GRAPH[ent]
    verbs = list(dat.keys())
    return verbs

def get_ent_encs(ents):
    ''' Gets the sentence transformer encodings for a list of entities '''
    return {e:st_model.encode(re.sub(r'[0-9]+', '', e)) for e in ents}


In [11]:
# Object to store the genome info
class FicGenome:
    def __init__(self, story_file:str, mc:str, ent:dict={}, verbs:dict={}):
        '''
            mc:    str
            ent:   {og_story_id: fic_rep_ent}
            verbs: {line: fic_rep_verb} (associated with line number in original story and verb_set)
        '''
        self.story_file = story_file

        # entities and main character
        self.mc = mc
        self.mc_dat = get_assoc_dat(mc)
        self.ent = ent
        self.ent_encs = get_ent_encs(list(ent.values()))

        # verbs
        self.verbs = verbs

        # algorithm properties
        self.fitness = 0
        self.fit_set = {'intra':0, 'inter':0, 'ent':0, 'verb':0}
        self.genome = self.make_genome()


    # ----- MUTATION METHODS ----- #
    def assign_new_ents(self, story, form='random'):
        ''' Generates a new set of entities for the FicGenome object
            form: 'random' | 'assoc'
        '''
        new_ent = {}

        # assume the main character is already set
        mc_subjs = self.mc_dat['subj'][:] if self.mc_dat and form == 'assoc' else []
        mc_objs = self.mc_dat['obj'][:] if self.mc_dat and form == 'assoc' else []

        mc_subjs = list(set(mc_subjs) - {self.mc})  # remove the MC from associated subjects
        mc_objs = list(set(mc_objs) - {self.mc})    # remove the MC from associated objects

        random.shuffle(mc_subjs)
        random.shuffle(mc_objs)

        # reassign entities based on the associative data
        saved_ents = {}     # class symbol : {"ent": entity, 'ct': count}
        mc_symb = story.mc_ent[1]
        saved_ents[mc_symb] = {'ent':self.mc, 'ct':1}   # always assign the MC to its symbol


        local_subjs = ALL_SUBJS[:]  # copy to avoid modifying the original list
        if self.mc and self.mc in local_subjs:
            local_subjs.remove(self.mc)  # remove the main character from the subject list

        local_objs = ALL_OBJS[:]  # copy to avoid modifying the original list
        if self.mc and self.mc in local_objs:
            local_objs.remove(self.mc)  # remove the main character from the object list
        
        for ent_id, ent_type in story.ent_ids.items():
            symbol = ent_id[1]   # get the symbol (e.g. A, b, 7, $, &, etc)

            if ent_id == story.mc_ent and self.mc is not None:      # assign the main character
                new_ent[ent_id] = self.mc
            elif symbol in saved_ents:                              # Reuse previously assigned entity if available
                new_ent[ent_id] = saved_ents[symbol]['ent'] + f"{saved_ents[symbol]['ct']+1}" #if ent_id != story.mc_ent else self.mc
                saved_ents[symbol]['ct'] += 1
                continue
            elif ent_type == 'subject':                             # assign subject entity
                if self.mc_dat and len(mc_subjs) > 0:
                    new_ent[ent_id] = mc_subjs.pop()
                else:
                    new_ent[ent_id] = random.choice(local_subjs)      # out of subject entities or random
            else:             # assign object entity    
                if self.mc_dat and len(mc_objs) > 0:
                    new_ent[ent_id] = mc_objs.pop()
                else:
                    new_ent[ent_id] = random.choice(local_objs)      # out of object entities

            if new_ent[ent_id] in local_subjs:  
                local_subjs.remove(new_ent[ent_id])     # remove copies of the newly added entity

            if new_ent[ent_id] in local_objs:
                local_objs.remove(new_ent[ent_id])      # remove copies of the newly added entity

            saved_ents[symbol] = {'ent':new_ent[ent_id], 'ct':1}  # save the assigned entity for this symbol

        # assign new entities
        self.ent = new_ent
        self.ent_encs = get_ent_encs(list(new_ent.values()))

    def assign_new_verbs(self, story, form='random', debug=False):
        ''' Generates a new set of verbs for the FicGenome object
            form: 'random' | 'assoc'
        '''
        if debug:
            print(self.ent)

        new_verbs = {}
        for line, (subj_ent, og_verb) in story.verb_set.items():
            if form == 'random':        # assign random verb
                new_verbs[line] = random.choice(ALL_VERBS)
            elif form == 'assoc':       # assign associated verb to noun
                assoc_verbs = get_assoc_verbs(self.ent[subj_ent]) if subj_ent in self.ent and story.ent_ids[subj_ent] == 'subject' else []
                if len(assoc_verbs) > 0:
                    new_verbs[line] = random.choice(assoc_verbs)
                else:
                    new_verbs[line] = random.choice(ALL_VERBS)
        self.verbs = new_verbs

    def hard_mutate(self, story, mc_form='random', ent_form='random', verb_form='random'):
        ''' Mutates the FicGenome object COMPLETELY
            mc:    'random' | 'same'
            ent:   'random' | 'assoc'
            verbs: 'random' | 'assoc'
        '''
        # TODO: mutate based on associations
        self.genome = None # reset genome

        if mc_form == 'random':
            self.mc = random.choice(ALL_SUBJS)
            self.mc_dat = get_assoc_dat(self.mc)

        self.assign_new_ents(story, form=ent_form)      # assign new entities (random or associated)
        self.assign_new_verbs(story, form=verb_form)     # assign new verbs (random or associated)

        # remake the genome based on new values
        self.genome = self.make_genome()


    def mutate(self, story, mut_chance=0.25, ent_form='random', verb_form='random', debug=False):
        ''' Keeps the main character the same but any verb or entity has a chance to be randomly replaced 
            Does not use associated entities
        
        '''

        mc_subjs = self.mc_dat['subj'][:] if self.mc_dat and ent_form == 'assoc' else []
        mc_objs = self.mc_dat['obj'][:] if self.mc_dat and ent_form == 'assoc' else []

        local_subjs = ALL_SUBJS[:]  # copy to avoid modifying the original list
        if self.mc and self.mc in local_subjs:
            local_subjs.remove(self.mc)  # remove the main character from the subject list

        local_objs = ALL_OBJS[:]  # copy to avoid modifying the original list
        if self.mc and self.mc in local_objs:
            local_objs.remove(self.mc)  # remove the main character from the object list

        random.shuffle(local_subjs)
        random.shuffle(local_objs)

        # get all the unique entities and their associated ids
        unique_ents = {}
        for id, ent in self.ent.items():
            class_ent = re.sub(r'[0-9]+', '', ent)  # remove numbers from the entity
            if class_ent not in unique_ents:
                unique_ents[class_ent] = {'ids':[], 'ent_type': story.ent_ids[id]}
            elif unique_ents[class_ent]['ent_type'] == 'object' and story.ent_ids[id] == 'subject': # override the type
                unique_ents[class_ent]['ent_type'] = 'subject'

            if id != self.mc:
                unique_ents[class_ent]['ids'].append(id)

        # change entity groups
        changed_ent = []
        new_ent = {}
        for class_ent, info in unique_ents.items():
            # change the entity group
            if random.random() < mut_chance:
                if info['ent_type'] == 'subject':
                    ne = mc_subjs.pop() if ent_form == 'assoc' and len(mc_subjs) > 0 else random.choice(local_subjs)
                    for i in range(len(info['ids'])):
                        new_ent[info['ids'][i]] = ne + f"{i+1}" if i > 0 else ne
                else:
                    ne = mc_objs.pop() if ent_form == 'assoc' and len(mc_objs) > 0 else random.choice(local_objs)
                    for i in range(len(info['ids'])):
                        new_ent[info['ids'][i]] = ne + f"{i+1}" if i > 0 else ne
                changed_ent.append(info['ids'][0])
                
            # keep the same
            else:
                for i in range(len(info['ids'])):
                    new_ent[info['ids'][i]] = self.ent[info['ids'][i]]


        if debug:
            print(f"# of changed entity groups: {len(changed_ent)} / {len(unique_ents)}")
            for id in self.ent.keys():
                if self.ent[id] != new_ent[id]:
                    print(f"{id}: {self.ent[id]} -> {new_ent[id]}")
            print("")

        # change verbs
        new_verbs = {}
        changed_verbs = 0
        for line, (subj_ent, og_verb) in story.verb_set.items():
            if random.random() < mut_chance:  # if this verb is to be changed
                if verb_form == 'assoc':
                    assoc_verbs = get_assoc_verbs(self.ent[subj_ent]) if subj_ent in self.ent else []
                    if len(assoc_verbs) > 0:
                        new_verb = random.choice(assoc_verbs)
                    else:
                        new_verb = random.choice(ALL_VERBS)
                else:
                    new_verb = random.choice(ALL_VERBS)
                new_verbs[line] = new_verb
                changed_verbs += 1
            else:
                new_verbs[line] = self.verbs[line]  # keep the original verb if not changed

        if debug:
            print(f"# of changed verbs: {changed_verbs} / {len(self.verbs)}")
            for id in self.verbs.keys():
                if self.verbs[id] != new_verbs[id]:
                    print(f"{id}: {self.verbs[id]} -> {new_verbs[id]}")
            print("")

        # set and re-embed
        self.ent = new_ent
        self.verbs = new_verbs
        self.ent_encs = get_ent_encs(list(self.ent.values()))


    # ------ NOVELTY / EVOLUTION METHODS ------ #

    def clone(self):
        ''' Returns a separate copy of this object '''
        new_fic = FicGenome(self.story_file, self.mc, {k:v for k,v in self.ent.items()}, {k:v for k,v in self.verbs.items()})
        new_fic.fitness = self.fitness
        new_fic.genome = self.genome
        new_fic.fit_set = {k:v for k,v in self.fit_set.items()}
        return new_fic


    def eval(self, af_story, debug=False, internal_debug=False):
        ''' Evaluates the fitness of the FicGenome object 
            Fitness is based on:
                - Semantic closeness of each sentence
                - Semantic closeness of the story sentences
                - Semantic closeness of the entities to the main character
                - Semantic closeness of the verb selection to the original verbs
        '''

        # check if the correct story
        if af_story.log_file != self.story_file:
            print(f"Error: Story file mismatch. Expected {self.story_file}, got {af_story.log_file}")
            return 0
        
        # un-TODO: INTERESTINGNESS METRIC
        # interestingness_score = random.random()  # placeholder for now (between 0 and 1)

        # Intersentence cohesion score: Sentence-to-sentence cohesion score
        story_sentences = self.generate_story(af_story, out_file=None)
        # embeddings = st_model.encode(story_sentences, convert_to_tensor=True)
        embeddings = st_model.encode(story_sentences)

        # cohesion = average cosine similarity of consecutive pairs
        sims = []
        for i in range(len(embeddings) - 1):
            cos_sim = np.dot(embeddings[i], embeddings[i+1]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1]))
            sims.append(float(cos_sim))
            if internal_debug:
                print(f"\t- Inter Sentence Cohesion ({i},{i+1}): {float(cos_sim):.4f}")
        inter_sentence_cohesion_score = float(np.mean(sims)) if sims else 0.0

        if debug:
            print(f"- Inter Cohesion: {inter_sentence_cohesion_score:.4f}")

        # Intrasentence cohesion score: Within each sentence cohesion score
        intra_scores = []
        for sent in story_sentences:
            words = [tok.strip("[]") for tok in sent.replace("]", " [").split() if tok.strip()]

            embeddings = st_model.encode(words)
            # compute all pairwise similarities
            sims = []
            for i in range(len(embeddings)):
                for j in range(i + 1, len(embeddings)):
                    cos_sim = np.dot(embeddings[i], embeddings[j]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))
                    sims.append(float(cos_sim))
            
                    if internal_debug:
                        print(f"\t- Inter Sentence Cohesion ({i},{j}): {float(cos_sim):.4f}")
                    
            intra_scores.append(float(np.mean(sims)))
            

        intra_cohesion = float(np.mean(intra_scores)) if intra_scores else 0.0
        if debug:
            print(f"- Intra Cohesion: {intra_cohesion:.4f}")


        # ENT SIMILARITY METRIC
        if len(self.ent_encs) == 0 or af_story.mc_ent not in self.ent:
            ent_score = 0
        else:
            mc_enc = st_model.encode(self.mc)
            cos_sims = []
            for e, enc in self.ent_encs.items():
                if e == self.mc:        # skip the main character
                    continue
                cos_sim = np.dot(mc_enc, enc) / (np.linalg.norm(mc_enc) * np.linalg.norm(enc))
                cos_sims.append(cos_sim)

                if internal_debug:
                    print(f"\t- MC Ent: {self.mc} | Fic Ent: {e} | Cosine Sim: {float(cos_sim):.4f}")
            ent_score = float(np.mean(cos_sims))    # between 0? and 1
        if debug:
            print(f"- Ent Score: {ent_score:.4f}")


        # VERB SIMILARITY METRIC
        
        # get encodings for verbs
        og_verbs = list(af_story.verb_set.values())
        fic_verbs = list(self.verbs.values())
        og_verb_vecs = [af_verb_enc_dict[v[1]] for v in og_verbs]
        fic_verb_vecs = [st_model.encode(v) for v in fic_verbs]

        # get cosine similarity between verb sets
        cos_sims = []
        for i in range(len(og_verb_vecs)):
            cos_sim = np.dot(og_verb_vecs[i], fic_verb_vecs[i]) / (np.linalg.norm(og_verb_vecs[i]) * np.linalg.norm(fic_verb_vecs[i]))
            cos_sims.append(cos_sim)

            if internal_debug:
                print(f"\t- OG Verb: {og_verbs[i][1]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")

        verb_score = float(np.mean(cos_sims)) # between 0? and 1
        if debug:
            print(f"- Verb Score: {verb_score:.4f}")

        # set the fitness
        # self.fitness = (intra_cohesion + inter_sentence_cohesion_score + interestingness_score + ent_score + verb_score) / 5.0
        self.fitness = (intra_cohesion + inter_sentence_cohesion_score + ent_score + verb_score) / 4.0

        self.fit_set = {
            'intra': intra_cohesion,
            'inter': inter_sentence_cohesion_score,
            'ent': ent_score,
            'verb': verb_score
        }

        return self.fitness

    def make_genome(self):
        ''' Creates a representation of the genome (ents+verbs) 
            Uses a sentence embedding model from sentence-transformers to convert the genome to a vector
            for comparison with other genomes.
        '''
        all_words = list(self.ent.values()) + list(self.verbs.values())
        all_words = ' '.join(all_words)
        genome = st_model.encode(all_words)
        return genome



    # ------ FILE I/O ------ #

    def generate_story(self, af_story, out_file:str=None):
        ''' Creates a story log based on the genome '''
        new_story = []
        for i, og_line in enumerate(af_story.og_text):
            new_line = og_line
            
            # Replace entities in the original line with their representations
            for ent_id, fic_rep in self.ent.items():
                new_line = new_line.replace(ent_id, f"[{fic_rep}]")

            # Replace verbs in the original line with their representations
            if i in self.verbs:
                af_story_verb = af_story.verb_set[i]
                fic_verb = self.verbs[i]
                new_line = new_line.replace(af_story_verb[1], fic_verb)

            new_story.append(new_line)

        # Write the modified line to the output file
        if out_file is not None:
            with open(out_file, 'w') as f:
                for line in new_story:
                    f.write(line + '\n')

        return new_story
    

    def export_fic(self, af_story, out_file:str=None):
        ''' Exports the data as a JSON file to reimport later '''
        fic_data = {
            'mc': self.mc,
            'ent': self.ent,
            'verbs': self.verbs,
            'fitness': self.fitness,
            'fit_set': self.fit_set,
            'story_file': self.story_file,
            'new_story': '\n'.join(self.generate_story(af_story)) if af_story is not None else None
        }
        if out_file is not None:
            with open(out_file, 'w') as f:
                json.dump(fic_data, f, indent=3)
        return fic_data

    def import_fic(self, dat):
        ''' Imports the data from a JSON file '''
        self.mc = dat['mc']
        self.ent = dat['ent']
        self.verbs = dat['verbs']
        self.fitness = dat['fitness']
        self.fit_set = dat['fit_set']
        self.story_file = dat['story_file']
        self.genome = self.make_genome()

### Helper Functions

In [12]:
def init_population(size, story, main_char='random', others='random'):
    ''' 
        Initializes a population of FicGenome objects based on the given AF_Story object 
        main_char:    'random' | (specific entity representation)
        others: 'random' | 'assoc' (associated to mc)
    '''
    population = []

    for _ in range(size):
        # choose a random MC from the entities in the story
        if main_char != "random" and main_char not in ALL_SUBJS:
            print(f"\t!!!    WARNING    !!!! Main character [{main_char}] not a possible subject in the graph. Defaulting to random choice.")

        mc = random.choice(ALL_SUBJS) if main_char == 'random' else main_char
        fic = FicGenome(story.log_file, mc)

        fic.assign_new_ents(story, form=others)      # assign new entities associated with the MC
        fic.assign_new_verbs(story, form=others)     # assign new verbs associated

        population.append(fic)

    
    return population

In [13]:
# test generating a story
pop = init_population(1, castle_story, main_char='random', others='random')
fic = pop[0]
print(f"\nFic: MC={fic.mc} ({castle_story.mc_ent}), Ents={fic.ent}, Verbs={fic.verbs}")
print(fic.eval(castle_story, debug=True))
print("\n=====MUTATE=====")
print(fic.mc)
print(fic.mc_dat)
fic.mutate(castle_story, ent_form='assoc', verb_form='assoc', debug=True)
print(fic.eval(castle_story, debug=True))
print()


Fic: MC=americans ([T.73ab]), Ents={'[K.fc7c]': 'rainfall', '[T.d89f]': 'americans2', '[T.73ab]': 'americans', '[=.23f6]': 'controversy', '[G.3305]': 'cellar', '[$.1a6e]': 'sauna', '[T.76a3]': 'americans2', '[$.6d7b]': 'sauna2', '[$.9fe4]': 'sauna3', '[=.2486]': 'controversy2', '[$.1b0e]': 'sauna4', '[=.0cc4]': 'controversy3', '[$.0bdf]': 'sauna5', '[K.7479]': 'rainfall2', '[$.6666]': 'sauna6', '[$.6924]': 'sauna7', '[$.db0a]': 'sauna8', '[$.4709]': 'sauna9'}, Verbs={0: 'filtered', 1: 'performed', 2: 'alphabetized', 3: 'spelled', 4: 'related', 5: 'searched', 6: 'sprayed', 7: 'duplicated', 8: 'scratched', 9: 'attained', 10: 'jogged', 11: 'colored', 12: 'viewed', 13: 'populated', 14: 'delineated', 15: 'unpleasant', 16: 'circulated', 17: 'fathered', 18: 'accentuated', 19: 'exchanged', 20: 'erased', 21: 'practiced', 22: 'disposed', 23: 'clipped', 24: 'skilled', 25: 'coloured', 26: 'threw', 27: 'extracted', 28: 'divided', 29: 'cared'}
- Inter Cohesion: 0.4844
- Intra Cohesion: 0.2597
- Ent

In [14]:
def is_novel(x, archive, threshold=0.5, debug=False):
    ''' Determines if a FicGenome object is novel compared to an archive of FicGenome objects '''
    if len(archive) == 0:       # nothing in the archive yet, so it's novel!
        return True

    # Get the minimum distance to any genome in the archive
    min_distance = float('inf')
    distances = []
    for a in archive:
        distance = np.linalg.norm(x.genome - a.genome)
        distances.append(distance)
        if distance < min_distance:
            min_distance = distance

    if debug:
        print(f"Distances: {distances}")

    # If the minimum distance is greater than the threshold, the genome is novel
    return min_distance >= threshold


In [15]:
def export_archive(arx, out_file:str, story=None):
    ''' Exports the archive of FicGenome objects to a JSON file '''
    archive_data = [x.export_fic(story) for x in arx]
    with open("../novelty_search_out/archive/"+out_file, 'w') as f:
        json.dump(archive_data, f, indent=3)

### Execute Novelty Search (Pure)

In [16]:
def novelty_search(af_log, params={}):
    ''' Main novelty search algorithm '''


    # get parameters and set defaults
    init_main_char = params.get('main_char', 'random')
    init_other_ents = params.get('other_ents', 'random')

    mut_chance = params.get('mut_chance', 0.25)
    mut_main_char = params.get('mut_main_char', 'random')
    mut_other_ents = params.get('mut_other_ents', 'random')
    mut_verbs = params.get('mut_verbs', 'random')

    fit_threshold = params.get('fit_threshold', 0.5)
    novel_threshold = params.get('novel_threshold', 0.5)
    rand_perc = params.get('rand_perc', 0.2)

    num_generations = params.get('num_generations', NUM_GENERATIONS)
    pop_size = params.get('pop_size', POP_SIZE)



    # 0. Initialize story representation
    story = AF_Story(af_log)
    
    # 1. Initialize population and archive
    population = init_population(POP_SIZE, story, main_char=init_main_char, others=init_other_ents)
    archive = []

    best_fitness = 0
    best_fic = None

    for gen in range(num_generations):
        print(f"Generation {gen+1} / {num_generations} -- [Overall Best Fitness: {best_fitness:.3f} | Archive Size: {len(archive)}]")

        # 8. Repeat 2-7 for NUM_GENERATIONS
        for indiv in population:

            # 2. Evaluate fitness
            indiv.eval(story)

            # 3+4. Evaluate novelty against archive and add if novel and fit enough
            if is_novel(indiv, archive, novel_threshold) and indiv.fitness > fit_threshold:
                archive.append(indiv.clone())


        # print some stats
        population.sort(key=lambda x: x.fitness, reverse=True)
        fit_scores = [indiv.fitness for indiv in population]
        print(f"  Pop Fitness: max {max(fit_scores):.3f}, min {min(fit_scores):.3f}, avg {sum(fit_scores)/len(fit_scores):.3f}")

        if gen % (num_generations // 20) == 0:
            print("   FicGenome of best population individual:")
            print(f"     - Best MC: {population[0].mc}")
            print(f"     - Best Ents: {list(population[0].ent.values())}")
            print(f"     - Best Verbs: {set(population[0].verbs.values())}")

        
        if population[0].fitness > best_fitness:
            best_fitness = population[0].fitness
            best_fic = population[0].clone()
            print(f"  New best fitness: {best_fitness:.3f}")

        # 5. Select new parents from novelty archive
        if len(archive) > 0:
            parents = random.choices(archive, k=int(pop_size*(1-rand_perc)))
        else:
            parents = random.choices(population, k=int(pop_size*(1-rand_perc)))

        # 6. Mutate children from parents
        new_pop = []
        for parent in parents:
            child = parent.clone()
            child.mutate(story, mut_chance=mut_chance, mc_form=mut_main_char, ent_form=mut_other_ents, verb_form=mut_verbs)
            new_pop.append(child)

        # 7. Add random individuals
        rand_amt = (pop_size - len(new_pop))
        for _ in range(int(pop_size*rand_perc)):
            randos = init_population(rand_amt, story, main_char=init_main_char, others=init_other_ents)
            new_pop.extend(randos)

        # update population
        population = new_pop

    return archive, best_fic, story

In [17]:
NOV_PARAMS = {
    'main_char': 'random',
    'other_ents': 'assoc',
    'mut_chance': 0.25,
    'mut_main_char': 'random',
    'mut_other_ents': 'assoc',
    'mut_verbs': 'assoc',
    'fit_threshold': 0.35,
    'novel_threshold': 0.5,
    'rand_perc': 0.2,
    'num_generations': 20,
    'pop_size': 10
}

arc, best_fic, story = novelty_search('../logs/stupid_log.txt', params=NOV_PARAMS)

Generation 1 / 20 -- [Overall Best Fitness: 0.000 | Archive Size: 0]
  Pop Fitness: max 0.333, min 0.239, avg 0.273
   FicGenome of best population individual:
     - Best MC: cinema
     - Best Ents: ['date', 'cinema', 'life', 'life2', 'film', 'money', 'movie', 'friend', 'movie', 'cyclist', 'money', 'date', 'friend', 'zoo', 'northerners']
     - Best Verbs: {'kept', 'stored', 'watched', 'caused', 'displayed', 'paid', 'laughed', 'chopped', 'reached'}
  New best fitness: 0.333


TypeError: FicGenome.mutate() got an unexpected keyword argument 'mc_form'. Did you mean 'ent_form'?

In [ ]:
# save the best fic story
t = datetime.now().strftime("[%m-%d-%Y %H%M]")
best_fic.export_fic(story, out_file=f'../novelty_search_out/fic_genomes/best_fic_log-{NOV_PARAMS['pop_size']}_{NOV_PARAMS['num_generations']}_{t}.json')
export_archive(arc, out_file=f'archive-{NOV_PARAMS['pop_size']}_{NOV_PARAMS['num_generations']}_{t}.json', story=story)

bf_story = best_fic.generate_story(story, out_file=f'../novelty_search_out/gen_stories/best_fic_log-{NOV_PARAMS['pop_size']}_{NOV_PARAMS['num_generations']}_{t}.txt')

print("best story:")
for line in bf_story:
    print(line)

best story:
=====    FORTRESS SEED: [791231]    =====
Fortress initialized! - <0>
>>> TIME: 2025-08-14 13:21:44 <<<
<0> [music] kept into [house]
<1> [orchestra] composed to [orchestra2] at (44, 2)
<2> [piano] performed to (65, 47)
<3> [hand] pushed to (65, 68)
<5> [orchestra] played into [string]
<6> [child] set by [string]
<7> [rag] restricted [hand] at (55, 7)
<8> [piano] played to (50, 78)
<9> [string] flew [tune]
<10> [house] located [rag]
<11> [sound] carried [rag] at (37, 88)
<12> [home] kept [music]


In [ ]:
best_fic.fit_set

{'intra': 0.3030000796464701,
 'inter': 0.372127819274153,
 'ent': 0.38659581542015076,
 'verb': 0.3512124717235565}

### MAP-Elites Search Experiment

In [ ]:
def map_elites(af_log, params={}):
    ''' Main MAP-Elites algorithm '''


    # get parameters and set defaults
    init_main_char = params.get('main_char', 'random')
    init_other_ents = params.get('other_ents', 'random')

    mut_chance = params.get('mut_chance', 0.25)
    mut_main_char = params.get('mut_main_char', 'random')
    mut_other_ents = params.get('mut_other_ents', 'random')
    mut_verbs = params.get('mut_verbs', 'random')

    rand_perc = params.get('rand_perc', 0.2)
    arx_cell_size = params.get('arx_cell_size', 5)

    num_generations = params.get('num_generations', NUM_GENERATIONS)
    pop_size = params.get('pop_size', POP_SIZE)



    # 0. Initialize story representation
    story = AF_Story(af_log)
    
    # 1. Initialize population and archive
    population = init_population(POP_SIZE, story, main_char=init_main_char, others=init_other_ents)
    archive = {}        # based on best fit_set (inter, intra, noun, verb cohesions)

    best_fitness = 0
    best_fic = None

    for gen in range(num_generations):
        arx_fit = {f:f"{s[0].fit_set[f]:.3f} [fit={s[0].fitness:.3f}]" for f,s in archive.items()} if len(archive) > 0 else {}
        print(f"Generation {gen+1} / {num_generations} \n-- [Overall Best Fitness: {best_fitness:.3f} | Archive: {arx_fit}]\n")

        # 8. Repeat 2-7 for NUM_GENERATIONS
        for indiv in population:

            # 2. Evaluate fitness
            indiv.eval(story)

            # 3+4. Evaluate fitness set scores and assign to archive
            fit_set = indiv.fit_set
            for f,v in fit_set.items():
                if f not in archive:            # initialize archive cell
                    archive[f] = []
                if len(archive[f]) == 0 or v > archive[f][-1].fit_set[f]:       # add if better than last element in the list
                    archive[f].append(indiv.clone())

                # sort and remove the lowest fitness
                archive[f] = sorted(archive[f], key=lambda x: x.fit_set[f], reverse=True)[:arx_cell_size]


        # print some stats
        population.sort(key=lambda x: x.fitness, reverse=True)
        fit_scores = [indiv.fitness for indiv in population]
        print(f"  Pop Fitness: max {max(fit_scores):.3f}, min {min(fit_scores):.3f}, avg {sum(fit_scores)/len(fit_scores):.3f}")

        if gen % (num_generations // 20) == 0:
            print("   FicGenome of best population individual:")
            print(f"     - Best MC: {population[0].mc}")
            print(f"     - Best Ents: {list(population[0].ent.values())}")
            print(f"     - Best Verbs: {set(population[0].verbs.values())}")

        
        if population[0].fitness > best_fitness:
            best_fitness = population[0].fitness
            best_fic = population[0].clone()
            print(f"  New best fitness: {best_fitness:.3f}")

        # 5. Select new parents from novelty archive
        if len(archive) > 0:
            arx_stories = [s for sublist in archive.values() for s in sublist]
            parents = random.choices(arx_stories, k=int(pop_size*(1-rand_perc)))
        else:
            parents = random.choices(population, k=int(pop_size*(1-rand_perc)))

        # 6. Mutate children from parents
        new_pop = []
        for parent in parents:
            child = parent.clone()
            child.mutate(story, mut_chance=mut_chance, mc_form=mut_main_char, ent_form=mut_other_ents, verb_form=mut_verbs)
            new_pop.append(child)

        # 7. Add random individuals
        rand_amt = (pop_size - len(new_pop))
        for _ in range(int(pop_size*rand_perc)):
            randos = init_population(rand_amt, story, main_char=init_main_char, others=init_other_ents)
            new_pop.extend(randos)

        # update population
        population = new_pop

    return archive, best_fic, story

In [ ]:
def export_me_archive(arx, out_file:str, story=None):
    ''' Exports the archive of FicGenome objects (from MAP-Elites experiment) to a JSON file '''
    archive_data = {}
    for k, v in arx.items():
        archive_data[k] = [x.export_fic(story) for x in v]
    with open("../map_elites_out/archive/"+out_file, 'w') as f:
        json.dump(archive_data, f, indent=3)

In [ ]:
ME_PARAMS = {
    'main_char': 'random',
    'other_ents': 'assoc',
    'mut_chance': 0.25,
    'mut_main_char': 'random',
    'mut_other_ents': 'assoc',
    'mut_verbs': 'assoc',
    'arx_cell_size': 5,
    'rand_perc': 0.2,
    'num_generations': 20,
    'pop_size': 10
}

arc, best_fic, story = map_elites('../logs/stupid_log.txt', params=ME_PARAMS)

Generation 1 / 20 
-- [Overall Best Fitness: 0.000 | Archive: {}]

  Pop Fitness: max 0.314, min 0.245, avg 0.272
   FicGenome of best population individual:
     - Best MC: calculator
     - Best Ents: ['light', 'top', 'number', 'number2', 'calculator', 'math', 'clowns', 'plants', 'mathematic', 'bacon', 'number', 'light', 'calculation', 'bracelet', 'twitter']
     - Best Verbs: {'wore', 'found', 'caused', 'organized', 'identified', 'planted', 'called', 'made', 'screened', 'taught', 'added', 'sat'}
  New best fitness: 0.314
Generation 2 / 20 
-- [Overall Best Fitness: 0.314 | Archive: {'intra': '0.306 [fit=0.309]', 'inter': '0.326 [fit=0.314]', 'ent': '0.330 [fit=0.314]', 'verb': '0.325 [fit=0.281]'}]

  Pop Fitness: max 0.340, min 0.251, avg 0.273
   FicGenome of best population individual:
     - Best MC: plate
     - Best Ents: ['bone', 'table', 'food', 'food2', 'plate', 'kitchen', 'meal', 'utensil', 'meal', 'cookie', 'bone', 'kitchen', 'food', 'chapel', 'joggers']
     - Best Verbs

In [ ]:
# save the best fic story
t = datetime.now().strftime("[%m-%d-%Y %H%M]")      # time
f = f"{ME_PARAMS['pop_size']}_{ME_PARAMS['num_generations']}_{ME_PARAMS['arx_cell_size']}_{t}"       # experiment data

best_fic.export_fic(story, out_file=f'../map_elites_out/fic_genomes/best_fic_log-{f}.json')
export_me_archive(arc, out_file=f'archive-{f}.json', story=story)
bf_story = best_fic.generate_story(story, out_file=f'../map_elites_out/gen_stories/best_fic_log-{f}.txt')

print("--- Best Story ---")
for line in bf_story:
    print(line)

--- Best Story ---
=====    FORTRESS SEED: [791231]    =====
Fortress initialized! - <0>
>>> TIME: 2025-08-14 13:21:44 <<<
<0> [case] tried into [victim]
<1> [criminal] visited to [criminal2] at (44, 2)
<2> [police] hurried to (65, 47)
<3> [people] prepared to (65, 68)
<5> [criminal] picked into [gun]
<6> [driver] started by [driver]
<7> [crime] took [people] at (55, 7)
<8> [police] carried to (50, 78)
<9> [gun] injured [crime]
<10> [victim] reported [grief]
<11> [car] suffered [crime] at (37, 88)
<12> [nerves] creamed [case]
